
# Jointures et agrégation de données avec Python

Ce document présente les techniques avancées de jointure et d'agrégation de données avec pandas pour analyser les données RH hospitalières, en se concentrant sur l'analyse des absences et du temps de travail.

La période d'étude s'étend du 1er septembre 2024 au 31 août 2025.

Cette session vous donne un exemple concret d'analyse efficace des données RH complexes et de création de rapports insights pour la prise de décision managériale.

## Objectifs de la séance

À la fin de cette séance, vous serez capable de :

1. Maîtriser les jointures avec pandas
2. Combiner efficacement plusieurs tables de données 
3. Effectuer des agrégations complexes avec groupby
4. Créer des premiers rapports statistiques complets 


# Introduction aux jointures avec pandas

## Types de jointures

pandas offre plusieurs types de jointures via la fonction `merge()` :

- `inner`: Garde uniquement les lignes présentes dans les deux tables
- `left`: Garde toutes les lignes de la table de gauche
- `right`: Garde toutes les lignes de la table de droite
- `outer`: Garde toutes les lignes des deux tables

Lorsqu'une ligne n'a pas de correspondance, pandas remplit les valeurs manquantes avec NaN.

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

## Syntaxe de base de merge()

Sous forme de fonction, la syntaxe de `pd.merge()` est la suivante :

```python
# Syntaxe générale
resultat = pd.merge(
    df_gauche, 
    df_droite,
    on='colonne_commune',        # Colonne de jointure commune
    how='inner',                 # Type de jointure
    left_on='col_gauche',        # Si noms différents
    right_on='col_droite',       # Si noms différents
    suffixes=('_x', '_y')        # Suffixes pour colonnes en conflit
)
```

Sous forme de méthode, on peut aussi écrire :
```python
# Syntaxe méthode
resultat = df_gauche.merge(
    df_droite,
    on='colonne_commune',
    how='inner',
    left_on='col_gauche',
    right_on='col_droite',
    suffixes=('_x', '_y')
)
```


# Chargement et préparation des données

## Import des tables principales

Importer les tables `personnes`, `contrats`, `emplois`, `absences`, `absences_types` et `conges_rtt` et convertir les dates au format datetime.

In [3]:
path="C:/Users/thiba/OneDrive/Documents/Master Aix-Marseille/Data Manipulation/Données CSV"
# Charger les tables de base
personnes = pd.read_csv(f'{path}/personnes.csv')
contrats = pd.read_csv(f'{path}/contrats.csv')
emplois = pd.read_csv(f'{path}/emplois.csv')
absences = pd.read_csv(f'{path}/absences.csv')
absences_types = pd.read_csv(f'{path}/absences_types.csv')
conges_rtt = pd.read_csv(f'{path}/conges_rtt.csv')

# Conversion des dates
contrats['debut'] = pd.to_datetime(contrats['debut'])
contrats['fin'] = pd.to_datetime(contrats['fin'])
absences['debut'] = pd.to_datetime(absences['debut'])
absences['fin'] = pd.to_datetime(absences['fin'])
conges_rtt['debut'] = pd.to_datetime(conges_rtt['debut'])
conges_rtt['fin'] = pd.to_datetime(conges_rtt['fin'])

print("Tables chargées :")
print(f"- Personnes: {len(personnes)} lignes")
print(f"- Contrats: {len(contrats)} lignes") 
print(f"- Emplois: {len(emplois)} lignes")
print(f"- Absences: {len(absences)} lignes")
print(f"- Types d'absences: {len(absences_types)} lignes")
print(f"- Congés/RTT: {len(conges_rtt)} lignes")

Tables chargées :
- Personnes: 3000 lignes
- Contrats: 4450 lignes
- Emplois: 299 lignes
- Absences: 2586 lignes
- Types d'absences: 23 lignes
- Congés/RTT: 9769 lignes


## Chargement des données de temps dû

Compléter la fonction ci-dessous qui charge un ou plusieurs fichiers de temps dû mensuels et les combine en un seul DataFrame. 

- En l'absence de données obtenues, retourner un DataFrame vide.
- Convertir les colonnes `start` et `end` en format datetime. Elles ont été enregistrées au format ISO8601.
- Utiliser une structure de type `try/except` pour gérer les erreurs de chargement.
- Concaténer les DataFrames chargés avec `pd.concat()`.

In [4]:
import glob
import os

def charger_temps_du_mensuel(annee=None, mois=None, path="C:/Users/thiba/OneDrive/Documents/Master Aix-Marseille/Data Manipulation/temps_du_plages/temps_du_plages"):
    """
    Charge les fichiers de temps dû mensuels.
    
    Args:
        annee: Année spécifique (None = toutes)
        mois: Mois spécifique (None = tous)
        path: Chemin des fichiers de temps dû (par défaut "../data/temps_du_plages")
    
    Returns:
        DataFrame: Données de temps dû consolidées
    """
    # Construire le pattern de fichier
    if annee and mois:
        pattern = f"{path}/temps_du_{annee}_{mois:02d}.csv"
    elif annee:
        pattern = f"{path}/temps_du_{annee}_*.csv"
    else:
        pattern = f"{path}/temps_du_*.csv"

    fichiers = glob.glob(pattern)
    
    if not fichiers:
        print(f"Aucun fichier trouvé avec le pattern: {pattern}")
        return pd.DataFrame()
    
    print(f"Chargement de {len(fichiers)} fichiers de temps dû...")
    
    dataframes = []
    for fichier in sorted(fichiers):
        try:
            # Compléter ici pour remplir la liste dataframes
            fc = pd.read_csv(fichier)
            for f in ['start', 'end']:
                if f in fc.columns:
                    fc[f] = pd.to_datetime(fc[f], errors='coerce')
            dataframes.append(fc)
        except Exception as e:
            print(f"  KO Erreur avec {fichier}: {e}")
    
    if dataframes:
        temps_du_complet = pd.concat(dataframes, ignore_index=True)
        print(f"Total: {len(temps_du_complet)} lignes de temps dû")
        return temps_du_complet
    else:
        return pd.DataFrame()

# Charger quelques mois pour les exemples
temps_du = charger_temps_du_mensuel(2024, 10)  # Octobre 2024 seulement
temps_du = charger_temps_du_mensuel(2024, 11)
temps_du = charger_temps_du_mensuel(2024, 12)
temps_du = charger_temps_du_mensuel()

Chargement de 1 fichiers de temps dû...
Total: 15140 lignes de temps dû
Chargement de 1 fichiers de temps dû...
Total: 13501 lignes de temps dû
Chargement de 1 fichiers de temps dû...
Total: 14332 lignes de temps dû
Chargement de 12 fichiers de temps dû...
Total: 170134 lignes de temps dû


# Jointures de base entre tables

## Jointure contrats-personnes-emplois

À l'aide de la fonction `pd.merge()`, créer une table enrichie des contrats avec les informations des personnes et des emplois.

- Les colonnes à garder sont : `'id', 'debut', 'fin', 'type', 'part_temps', 'nom', 'prénom', 'date de naissance', 'genre', 'libellé', 'catégorie', 'rythme', 'id_pers', 'id_emploi'`
- Ajouter `age`, l'âge de la personne (en année, arrondi à l'unité) et `anciennete_jours`, `anciennete_annees` l'ancienneté du contrat (en jours sans arrondi pour le premier ; en années, arrondi à l'unité près pour le second). Ces quantités doivent être calculées à la fin de la période d'étude, c'est-à-dire au 31 août 2025.

In [5]:
def creer_table_contrats_enrichie():
    """
    Crée une table enrichie des contrats avec infos personnes et emplois.
    
    Returns:
        DataFrame: Contrats avec toutes les informations contextuelles
    """
    # Jointure contrats -> personnes et emplois
    resultat = pd.merge(
    contrats, 
    personnes,
    #on='id',        # Colonne de jointure commune
    how='inner',                 # Type de jointure
    left_on='id_pers',        # Si noms différents
    right_on='id',       # Si noms différents
)
    # Compléter ici pour créer la table contrats_enrichis
    contrats_enrichis = pd.merge(
    resultat, 
    emplois,
    #on='id',        # Colonne de jointure commune
    how='inner',                 # Type de jointure
    left_on='id_emploi',        # Si noms différents
    right_on='id',       # Si noms différents
)
    # Nettoyer les colonnes en doublon
    colonnes_a_garder = [
        'id', 'debut', 'fin', 'type', 'part_temps',
        'nom', 'prénom', 'date de naissance', 'genre',
        'libellé', 'catégorie', 'rythme',
        'id_pers', 'id_emploi'
    ]
    
    contrats_enrichis = contrats_enrichis[colonnes_a_garder]
    
    # Calculer l'âge et l'ancienneté
    contrats_enrichis['date de naissance'] = pd.to_datetime(contrats_enrichis['date de naissance'])
    contrats_enrichis['age'] = (contrats_enrichis['fin'] - contrats_enrichis['date de naissance']).dt.days # Corriger ici
    contrats_enrichis['age'] = contrats_enrichis['age']/365.25
    contrats_enrichis['anciennete_jours'] = (contrats_enrichis['fin'] - contrats_enrichis['debut']).dt.days# Corriger ici
    contrats_enrichis['anciennete_annees'] = contrats_enrichis['anciennete_jours']/365.25 # Corriger ici

    return contrats_enrichis

# Créer la table enrichie
contrats_enrichis = creer_table_contrats_enrichie()

print("Table contrats enrichie créée:")
print(f"Dimensions: {contrats_enrichis.shape}")
print("\nPremières lignes:")
print(contrats_enrichis[['nom', 'prénom', 'libellé', 'catégorie', 'type', 'age']].head())

Table contrats enrichie créée:
Dimensions: (4450, 17)

Premières lignes:
        nom    prénom                                            libellé  \
0   Alvarez     Maria                              Infirmier réanimation   
1  Belkacem  Ibrahima                                 Interprète médical   
2  Belkacem  Ibrahima                                 Interprète médical   
3   Nikolić      Hugo            Infirmier d'hémovigilance / transfusion   
4  Youssouf    Miguel  Chargé des affaires médicales (planning pratic...   

          catégorie type        age  
0  Soins infirmiers  cdi  19.550992  
1     Administratif  cdi  18.617385  
2     Administratif  cdi  19.362081  
3  Soins infirmiers  cdd  18.962355  
4     Administratif  cdi        NaN  


## Jointure avec types d'absences

Créer maintenant une fonction qui enrichit la table des absences avec les informations de `absences_types` et les informations de contrat précédemment enrichies. 

- Les colonnes à garder sont : `'id', 'id_contrat', 'debut', 'fin', 'jours_ouvres', 'code', 'libellé', 'libellé_contrat', 'famille', 'type', 'type_contrat', 'nom', 'prénom', 'catégorie', 'age'`
- Renommer `libellé` en `libellé_absence` et `type` en `type_absence`

En utilisant les méthodes `groupby`, `size`, `sort_values(ascending=False)` et `head`, afficher les 5 types d'absences les plus fréquents dans les lignes de votre tableau enrichi.

In [6]:
def enrichir_absences():
    """
    Enrichit les absences avec les types et informations contrats.
    
    Returns:
        DataFrame: Absences avec descriptions complètes
    """
    # Jointure absences -> types d'absences
    absences_typees = pd.DataFrame()  # Corriger ici 
    absences_typees = pd.merge(
    absences, 
    absences_types,
    on='code',        # Colonne de jointure commune
    how='inner',                 # Type de jointure
    suffixes=('', '_y')
    )
    absences_typees = absences_typees[[c for c in absences_typees.columns if not c.endswith('_y')]]
    # Jointure avec la vue contrats enrichie
    absences_enrichies = pd.DataFrame()  # Corriger ici
    absences_enrichies = pd.merge(absences_typees, contrats_enrichis, how='inner', left_on = 'id_contrat',
                                  right_on = 'id', suffixes=('', '_contrat'))
    # Sélectionner les colonnes utiles
    colonnes_finales = [
        'id', 'id_contrat', 'debut', 'fin', 'jours_ouvres',
        'code', 'libellé', 'libellé_contrat', 'famille', 'type', 'type_contrat',
        'nom', 'prénom', 'catégorie', 'age'
    ]
    
    # Renommer pour clarifier
    absences_enrichies = absences_enrichies[colonnes_finales].rename(columns={
        'libellé': 'libellé_absence',
        'type': 'type_absence'
    })
    absences_enrichies = absences_enrichies.loc[:,~absences_enrichies.columns.duplicated()].copy()
    return absences_enrichies

absences_enrichies = enrichir_absences()

print("Absences enrichies:")
print(f"Dimensions: {absences_enrichies.shape}")
print("\nExemple d'absences par type:")
print(absences_enrichies.groupby('libellé_absence').size().sort_values(ascending=False).head())
print(absences_enrichies)

Absences enrichies:
Dimensions: (4944, 15)

Exemple d'absences par type:
libellé_absence
Maladie                4122
Accident de travail     295
Enfant malade           244
Mariage/PACS             92
Déménagement             68
dtype: int64
       id  id_contrat      debut        fin  jours_ouvres code  \
0       1           5 2024-10-07 2024-10-13             5  MAL   
1       1           5 2024-10-07 2024-10-13             5  MAL   
2       1           5 2024-10-07 2024-10-13             5  MAL   
3       1           5 2024-10-07 2024-10-13             5  MAL   
4       1           5 2024-10-07 2024-10-13             5  MAL   
...   ...         ...        ...        ...           ...  ...   
4939  335         299 2025-08-17 2025-08-22             5  MAL   
4940  335         299 2025-08-17 2025-08-22             5  MAL   
4941  335         299 2025-08-17 2025-08-22             5  MAL   
4942  335         299 2025-08-17 2025-08-22             5  MAL   
4943  335         299 2025-08-17

# Jointures et agrégations complexes

L'objectif est maintenant d'étudier le taux d'absence, c'est-à-dire 
$$
\frac{\sum\text{durée de travail dû pendant absence}}{\sum\text{durée de travail dû}}.
$$

Il faut donc donc faire le lien entre les tables de temps dû et la table des absences. Une plage de temps dû est considérée comme absente si

(1) les identifiants de contrats correspondent et
(2) les périodes de temps se chevauchent.

La condition 1 revient à faire une jointure entre les tables de temps dû et d'absences sur l'identifiant de contrat. Et la condition 2 est un filtre sur cette jointure.

:::{.callout-important}
### Attention
Dans toute cette partie, si l'on s'y prend mal, on risque d'avoir des temps de calcul très longs. Il est donc recommandé de procéder par étape. 

- Ne charger qu'une seule table mensuelle de plages de temps dû dans votre DataFrame `temps_du` pendant que vous concevez et déboguez les différentes fonctions. 
- Une fois toutes les fonctions conçues et vérifiées, vous pourrez charger l'ensemble des tables mensuelles et réexécuter les calculs.
:::

In [7]:
def ajouter_identifiant_unique(temps_du):
    """
    Ajoute une colonne d'identifiant unique à la table des temps dû.
    
    Args:
        temps_du (DataFrame): Table des temps dû.
    
    Returns:
        DataFrame: Table des temps dû avec identifiant unique.
    """
    temps_du['id_temps_du'] = temps_du.index + 1 # Corriger ici
    return temps_du
temps_du = ajouter_identifiant_unique(temps_du)
print(temps_du)

           mois  id_pers  id_contrat  id_emploi  id_temps plage_type  \
0       2024-09        4           5        216       309    forfait   
1       2024-09        4           5        216       309    forfait   
2       2024-09        4           5        216       309    forfait   
3       2024-09        4           5        216       309    forfait   
4       2024-09        4           5        216       309    forfait   
...         ...      ...         ...        ...       ...        ...   
170129  2025-08     3000        4450         43        87       nuit   
170130  2025-08     3000        4450         43        87       nuit   
170131  2025-08     3000        4450         43        87       nuit   
170132  2025-08     3000        4450         43        87       nuit   
170133  2025-08     3000        4450         43        87       nuit   

                     start                 end  duree_heures   unite  \
0      2024-09-02 09:00:00 2024-09-02 17:00:00          8.00   

## Jointure absences et temps dû

L'objectif ici est de créer une table de temps dû pendant les absences. On ne garde une ligne que si elle est présente dans les deux tables. Ce qui revient à faire une jointure interne. 

Écrire une fonction `joindre_temps_absences` qui, à partir des tables de temps dû et la table `absences` non enrichies, effectue la jointure et le filtre.

In [8]:
def joindre_temps_absences(temps_du, absences):
    """
    Jointure entre les temps dû et les absences.

    Args:
        temps_du (DataFrame): Table des temps dû.
        absences (DataFrame): Table des absences.

    Returns:
        DataFrame: Table des temps dû pendant les absences.
    """
    # Jointure temps_du et absences sur l'identifiant de contrat
    jointure = pd.DataFrame()  # Corriger ici
    jointure = pd.merge(temps_du, absences, on='id_contrat', how='inner', suffixes = ('', '_abs'))
    # Filtrer les périodes de temps se chevauchant
    temps_absences = jointure[(jointure['start']<=jointure['fin']) & (jointure['end']>=jointure['debut'])]

    return temps_absences

temps_absences = joindre_temps_absences(temps_du, absences)

## Enrichissement de la table des plages horaires de temps dû

Écrire une fonction qui ajoute une colonne `absence` à la table `temps_du`. Cette colonne vaut $1$ s'il s'agit d'une absence, `0` sinon. 

In [9]:
def enrichir_temps_du(temps_du, temps_absences):
    """
    Enrichit la table des temps dû avec une colonne d'absence.

    Args:
        temps_du (DataFrame): Table des temps dû.
        temps_absences (DataFrame): Table des temps dû pendant les absences.

    Returns:
        DataFrame: Table des temps dû enrichie.
    """
    temps_du['absence'] = np.zeros(len(temps_du))
    ids_abs = temps_absences['id_contrat'].unique()
    temps_du.loc[temps_du['id_contrat'].isin(ids_abs), 'absence'] = 1 # Corriger ici
    return temps_du

temps_du = enrichir_temps_du(temps_du, temps_absences)
print(temps_du)

           mois  id_pers  id_contrat  id_emploi  id_temps plage_type  \
0       2024-09        4           5        216       309    forfait   
1       2024-09        4           5        216       309    forfait   
2       2024-09        4           5        216       309    forfait   
3       2024-09        4           5        216       309    forfait   
4       2024-09        4           5        216       309    forfait   
...         ...      ...         ...        ...       ...        ...   
170129  2025-08     3000        4450         43        87       nuit   
170130  2025-08     3000        4450         43        87       nuit   
170131  2025-08     3000        4450         43        87       nuit   
170132  2025-08     3000        4450         43        87       nuit   
170133  2025-08     3000        4450         43        87       nuit   

                     start                 end  duree_heures   unite  \
0      2024-09-02 09:00:00 2024-09-02 17:00:00          8.00   

## Préparation à l'agrégation pour analyse

Il reste à agréger cette table enrichies des plages de temps dû pour pouvoir procéder à l'analyse. On veut pouvoir utiliser différents filtres dans l'analyse :

- la période (mensuelle, trimestrielle, annuelle)
- la famille et le type d'absence
- le type de contrat (apprenti, cdd, cdi)
- la catégorie de l'emploi (Médical, Encadrement, Laboratoire, Pharmacie, etc.)
- la tranche d'âge (moins de 20 ans, 20-30 ans, 30-40 ans, 40-50 ans, 50-60 ans, 60 ans et plus)

<b>N.B.</b> L'âge est calculé à la date de fin de la période d'étude, c'est-à-dire au 31 août 2025.

Les variables d'agrégation sont donc : `mois`, `famille`, `type_absence`, `type_contrat`, `categorie`, `tranche_age`.

Écrire une fonction qui, éventuellement à l'aide de jointure gauche, enrichit la table des plages de temps dû avec les informations utiles et ajoute les colonnes nécessaires.

In [10]:
def continuer_enrichir_temps_du(temps_du, absences_enrichies, contrats_enrichis):
    """
    Enrichit la table des temps dû avec des informations supplémentaires pour l'analyse.

    Args:
        temps_du (DataFrame): Table des temps dû avec colonne absence.
        absences_enrichies (DataFrame): Table des absences enrichies.
        contrats_enrichis (DataFrame): Table des contrats enrichis.

    Returns:
        DataFrame: Table des temps dû enrichie avec toutes les variables d'analyse.
    """
    # Jointure gauche temps_du avec contrats pour récupérer type_contrat, catégorie et age
    temps_du_enrichi = pd.DataFrame()
    temps_du_enrichi = pd.merge(temps_du, contrats_enrichis, on='id_pers', how='left', suffixes = ('', '_y'))
    colonnes_finales = ['mois', 'id_pers', 'id_contrat', 'id_emploi', 'id_temps', 'plage_type', 
                        'start', 'end', 'duree_heures', 'unite', 'quota_source',  'id_temps_du', 
                        'absence', 'type', 'catégorie', 'age']
    temps_du_enrichi = temps_du_enrichi[colonnes_finales].rename(columns={'type' : 'type_contrat'})
    # Ajouter la colonne mois
    temps_du_enrichi['mois'] = np.ones(len(temps_du_enrichi), dtype=int)  # Corriger ici
    temps_du_enrichi['mois'] = temps_du_enrichi['end'].dt.to_period('M')
    # Ajouter la tranche d'âge
    def calculer_tranche_age(age):
        # Compléter et corriger ici
        if pd.isna(age):
            return 'Non défini'
        if age < 20:
            return 'Moins de 20 ans'
        elif age < 30:
            return '20-30 ans'
        elif age < 40:
            return '30-40 ans'
        elif age < 50:
            return '40-50 ans'
        elif age < 60:
            return '50-60 ans'
        else:
            return '60 ans et plus'
        
    
    temps_du_enrichi['tranche_age'] = temps_du_enrichi['age'].apply(calculer_tranche_age)
    # Jointure avec absences enrichies pour récupérer famille et type_absence
    temps_du_avec_absences = pd.DataFrame()  # Corriger ici
    temps_du_avec_absences = pd.merge(temps_du_enrichi, absences_enrichies, on='id_contrat', how='left', suffixes =('', '_y'))
    
    # Filtrer pour ne garder famille et type_absence que si chevauchement temporel
    chevauchement = ((temps_du_avec_absences['start'] <= temps_du_avec_absences['fin']) & (temps_du_avec_absences['end'] >= temps_du_avec_absences['debut']))
    temps_du_avec_absences.loc[~chevauchement, ['famille', 'type_absence']]
    colonnes_finales = ['mois', 'id_pers', 'id_contrat', 'id_emploi', 'id_temps', 'plage_type', 
                        'start', 'end', 'duree_heures', 'unite', 'quota_source',  'id_temps_du', 
                        'absence', 'type_contrat', 'catégorie', 'age', 'famille', 'type_absence', 'tranche_age']
    temps_du_avec_absences = temps_du_avec_absences[colonnes_finales]
    return temps_du_avec_absences

temps_du = continuer_enrichir_temps_du(temps_du, absences_enrichies, contrats_enrichis)

## Agrégations finales pour l'analyse

Il faut maintenant réaliser trois étapes :

- Agréger les plages de temps dû pour obtenir la somme des durées de ces plages par valeurs des variables de groupby. Cela donne une table des dénominateur du taux d'absentéisme.
- Réaliser la même agrégation, mais sur la table filtrée des plages avec absence. Cela donne une table des numérateurs du taux d'absentéisme.
- Joindre ces deux tables pour avoir les numérateurs et les dénominateurs. S'assurer qu'il n'y a pas de valeurs manquantes.

<b>N.B.</b> On ne calcule pas le rapport à ce stade car, si l'on regroupe deux groupes, par exemple les catégories `Encadrement` et `Direction`, pour former un nouveau groupe, on doit sommer numérateurs et dénominateurs correspondant pour avoir le taux d'absentéisme de ce nouveau groupe.

In [11]:
# Agrégation des dénominateurs (total temps dû)
variables_groupby = ['mois', 'famille', 'type_absence', 'type_contrat', 'catégorie', 'tranche_age']

# Remplacer les NaN par 'Aucune' pour les colonnes liées aux absences
temps_du_clean = temps_du.fillna({
    'famille': 'Aucune', 
    'type_absence': 'Aucune'
})

denominateurs = 0 # Corriger ici
denominateurs = temps_du_clean.groupby(variables_groupby, dropna=False)['duree_heures'].sum().reset_index()
denominateurs.rename(columns={'duree_heures': 'temps_du_total'}, inplace=True)

# Agrégation des numérateurs (temps dû pendant absences)
numerateurs = 0 # Corriger ici
df = temps_du_clean[temps_du_clean['absence'] > 0]
numerateurs = df.groupby(variables_groupby, dropna=False)['absence'].sum().reset_index()
numerateurs.rename(columns={'absence': 'temps_absence_total'}, inplace=True)
# Jointure pour obtenir numérateurs et dénominateurs
taux_absenteisme_data = pd.DataFrame()  # Corriger ici
taux_absenteisme_data = pd.merge(
    denominateurs,
    numerateurs,
    on=variables_groupby,
    how='left'
)

# Remplacer les NaN dans les numérateurs par 0 (pas d'absence)
taux_absenteisme_data['temps_absence_total'] = taux_absenteisme_data['temps_absence_total'].fillna(0)

print(f"Table d'analyse créée avec {len(taux_absenteisme_data)} lignes")
print("Premières lignes:")
print(taux_absenteisme_data.head())
print("dernieres lignes:")
print(taux_absenteisme_data.tail())

Table d'analyse créée avec 2761 lignes
Premières lignes:
      mois              famille         type_absence type_contrat  \
0  2024-09  Absences non payées               Divers          cdi   
1  2024-09       Accidentologie  Accident de travail          cdd   
2  2024-09       Accidentologie  Accident de travail          cdd   
3  2024-09       Accidentologie  Accident de travail          cdd   
4  2024-09       Accidentologie  Accident de travail          cdi   

               catégorie tranche_age  temps_du_total  temps_absence_total  
0  Administratif/médical  Non défini         4335.00                578.0  
1                Médical   20-30 ans         2448.00                306.0  
2                Médical   30-40 ans         2448.00                306.0  
3    Techniques/bâtiment   40-50 ans          660.48                 86.0  
4  Administratif/médical   20-30 ans        11424.00               1428.0  
dernieres lignes:
     mois               famille   type_absence type_co

## Un exemple de résultat

Écrire une fonction qui, à partir de la dernière table agrégée produite calcule les taux d'absentéisme par mois et par famille d'absence. On mettra les mois en ligne et les familles d'absence en colonnes.

In [12]:
def calculer_taux_absenteisme_mois_famille(taux_absenteisme_data):
    """
    Calcule les taux d'absentéisme par mois et famille d'absence.
    
    Args:
        taux_absenteisme_data (DataFrame): Données agrégées avec numérateurs et dénominateurs.
    
    Returns:
        DataFrame: Tableau croisé dynamique avec taux d'absentéisme (%).
    """
    # Agréger par mois et famille d'absence (sommer sur les autres dimensions)
    resume_mois_famille = pd.DataFrame()  # Corriger ici
    resume_mois_famille = (taux_absenteisme_data.groupby(['mois', 'famille'], dropna=False)[['temps_absence_total', 'temps_du_total']].sum().reset_index())
    # Calculer le taux d'absentéisme en pourcentage
    resume_mois_famille['taux_absenteisme'] = np.zeros(len(resume_mois_famille))  # Corriger ici
    resume_mois_famille['taux_absenteisme'] = (resume_mois_famille['temps_absence_total'] / resume_mois_famille['temps_du_total']) * 100
    # Créer un tableau croisé dynamique
    tableau_croise = pd.DataFrame()  # Corriger ici
    tableau_croise = resume_mois_famille.pivot_table(index='mois', columns='famille', values='taux_absenteisme', fill_value=0)
    return tableau_croise

# Exemple d'utilisation
exemple_taux = calculer_taux_absenteisme_mois_famille(taux_absenteisme_data)
print("Taux d'absentéisme par mois et famille d'absence (%):")
print(exemple_taux.round(2))

Taux d'absentéisme par mois et famille d'absence (%):
famille  Absences non payées  Accidentologie  Aucune  Autres  Maladie  \
mois                                                                    
2024-09                13.33           12.11    9.41   13.33    12.28   
2024-10                13.33           12.13    9.49   13.33    12.25   
2024-11                13.33           12.14    9.32   13.33    12.24   
2024-12                13.33           12.08    9.49   13.33    12.27   
2025-01                13.33           12.13    9.42   13.33    12.21   
2025-02                13.33           12.12    9.32   13.33    12.09   
2025-03                13.33           12.12    9.49   13.33    12.29   
2025-04                13.33           12.15    9.41   13.33    12.30   
2025-05                13.33           12.09    9.31   13.33    12.17   
2025-06                13.33           12.14    9.39   13.33    12.28   
2025-07                13.33           12.15    9.46   13.33    12.26 

# Exercice à rendre

Faire une analyse des périodes de vacances et de RTT des employés de l'hôpital *Santé Bonne Mère*. L'objectif est de savoir à quel moments ces périodes se concentrent typiquement pour éventuellement prévoir le recours à des interims. Les principales questions auxquelles ont souhaiterait répondre par cette analyse sont donc :

- Sur quelles périodes de l'année se concentrent ces périodes de vacances et de RTT ?
- Y a-t-il des différences notables entre les différents types de contrats (apprenti, cdd, cdi) ou les différentes catégories d'emploi ?

Il est inutile d'utiliser les plages de temps dû ici.

L'objectif du code est de créer une table préparant cette analyse.

**Livrables**

- le code commenté de création de la table à partir des fichiers CSV bruts
- une description courte des différentes étapes pour obtenir la table d'intérêt
- un exemple d'utilisation de cette table préparée pour commencer à répondre aux questions d'intérêt

**Éléments d'évaluation**

- Décomposition en étapes claires et logiques
- Justesse du code proposé
- Utilisation de fonctions pour encapsuler les différentes étapes de création de la table
- Commentaires du code (ce qu'il faut, mais juste ce qu'il faut)


# Annexe

## Concepts à maîtriser

1. **Jointures pandas** : merge() avec différents types (inner, left, right, outer)
2. **Agrégations complexes** : groupby avec fonctions multiples et métriques dérivées
3. **Analyse temporelle** : Évolution des indicateurs dans le temps
4. **Jointures multiples** : Combinaison de plusieurs tables pour analyses croisées
5. **Rapports de synthèse** : Création de tableaux de bord automatisés

## Fonctions pandas essentielles à apprendre

- **`pd.merge()`** : Jointures entre DataFrames
- **`groupby().agg()`** : Agrégations avec fonctions multiples
- **`fillna()`** : Remplissage des valeurs manquantes
- **`pivot()`** : Reformatage de données tabulaires
- **`pd.concat()`** : Concaténation de DataFrames
- **`.dt`** : Accesseur pour opérations sur dates

## Techniques d'analyse avancées

- **Jointures en cascade** : Chaînage de plusieurs merge()
- **Métriques dérivées** : Calcul d'indicateurs à partir d'agrégations
- **Analyse comparative** : Confrontation de différentes sources
- **Gestion des valeurs manquantes** : `fillna()` dans les jointures
- **Performance** : Optimisation des requêtes sur gros volumes


In [13]:
def preparer_table_conges(conges_rtt, contrats_enrichis):
    """
    Prépare la table d'analyse des congés et RTT.

    Args:
        conges_rtt (DataFrame): Table des congés et RTT.
        contrats_enrichis (DataFrame): Table des contrats enrichis.

    Returns:
        DataFrame: Table des jours de congés enrichie pour analyse.
    """
    # 1. Jointure congés et contrats_enrichis
    conges_enrichis = pd.merge(
        conges_rtt,
        contrats_enrichis,
        left_on='id_contrat',
        right_on='id',
        how='inner',
        suffixes=('', '_contrat')
    )
    conges_enrichis = conges_enrichis.loc[:, ~conges_enrichis.columns.duplicated()].copy() # Supprime les colonnes
                                                                                            # double

    # 2. Calcul de la duree des congés et du mois des congés

    conges_enrichis['duree'] = (conges_enrichis['fin'] - conges_enrichis['debut']).dt.days + 1
    conges_enrichis['mois'] = conges_enrichis['debut'].dt.month

    # 3. Sélection des colonnes utiles
    colonnes_finales = ['mois', 'code', 'type', 'catégorie', 'duree']
    table_conges = conges_enrichis[colonnes_finales]
    return table_conges

# Exemples d'utilisation avec aggrégations et tableaux croises dynamiques
table_conges = preparer_table_conges(conges_rtt, contrats_enrichis)
print(f"Table des durées de congés crée avec {len(table_conges)} lignes")
print(table_conges.head())

conges_par_mois = conges_enrichis.groupby(['mois'])['duree'].sum()
print("Jours de congés par mois :")
print(conges_par_mois)

conges_par_type = conges_enrichis.groupby(['type'])['duree'].sum()
print("Jours de congés par type de contrat :")
print(conges_par_type)

conges_par_categorie = conges_enrichis.groupby(['catégorie'])['duree'].sum()
print("Jours de congés par catégorie :")
print(conges_par_categorie)

tableau_type = conges_enrichis.groupby(['mois', 'type'])['duree'].sum()
print("Jours de congés par mois et par type de contrat :")
print(tableau_type)

tableau_cat = conges_enrichis.groupby(['mois', 'catégorie'])['duree'].sum()
print("Jours de congés par mois et par catégorie :")
print(tableau_cat)

Table des durées de congés crée avec 11906 lignes
   mois code type catégorie  duree
0     9  RTT  cdi   Médical      2
1     9  RTT  cdi   Médical      2
2     9  RTT  cdd   Médical      2
3     9  RTT  cdi   Médical      2
4     9  RTT  cdi   Médical      2


NameError: name 'conges_enrichis' is not defined